# INJECTION-R4c: IT Upsampling Fix

**Why R4b failed:** IT domain F1 = 0.7611 (gate > 0.80). Finance = 0.7901. Non-IT = 0.7707.  
IT is the only domain that regressed vs R4 on direct gate comparison (-0.013).  
Root cause: MiniLM encoder needs more IT training exposure; class weighting alone is insufficient.

**Fix -- IT 2x upsampling + adjusted class weights:**
- IT training samples duplicated 2x in both Phase 1 and Phase 2 (genuine data exposure, not synthetic)
- Finance class weight increased (still failing at 0.790, needs ~+0.010 to clear gate)
- Input weights: r4b_final_weights.h5 (best available baseline: Dom F1=0.8369)

### Final Hard Gates -- ALL must pass
| Metric | Gate |
|--------|------|
| ATS val MAE (0-100) | < 6.5 |
| ATS test MAE (0-100) | < 7.0 |
| Domain F1 macro | > 0.85 |
| Domain F1 per-class (all 7) | > 0.80 each |
| RSG val Accuracy | > 0.65 |
| Fresher fairness gap | <= 20 pts |

### Required files -- upload to `ATS_R4c/` on Google Drive
```
MyDrive/ATS_R4c/
  ats_tokenized.npz          <- from data/tokenized/
  rsg_tokenized.npz          <- from data/tokenized/
  data_splits.json           <- from data/tokenized/
  r4b_final_weights.h5       <- from ATS_R4b/output/ (R4c baseline)
  merged_final.csv           <- from data/labeled/  (optional: fresher fairness)
```
> **Runtime:** T4 GPU -- `Runtime -> Change runtime type -> T4 GPU`

In [1]:
# Cell 1: GPU check
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('No GPU! Go to Runtime -> Change runtime type -> T4 GPU then re-run.')
else:
    print(result.stdout)

Sun May 10 07:21:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Install dependencies
!pip install -q transformers==4.44.2 scikit-learn
print('Done.')
print('>>> ACTION: Runtime -> Restart session -> re-run all cells from Cell 1 <<<')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 82.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 148.1 MB/s eta 0:00:00
Done.
>>> ACTION: Runtime -> Restart session -> re-run all cells from Cell 1 <<<


In [3]:
# Cell 3: Mount Drive & configure paths
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
#  CONFIGURE: your ATS_R4c folder on Drive
# ================================================================
DRIVE_ROOT = '/content/drive/MyDrive/ATS_R4c'

import os
from pathlib import Path

DATA_DIR     = Path(DRIVE_ROOT)
IN_WEIGHTS   = str(DATA_DIR / 'r4b_final_weights.h5')   # input: R4b output (R4c baseline)
SPLITS_JSON  = str(DATA_DIR / 'data_splits.json')
MERGED_CSV   = str(DATA_DIR / 'merged_final.csv')        # optional

OUT_DIR      = DATA_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)
P1_BEST      = str(OUT_DIR / 'r4c_phase1_best.h5')
P2_BEST      = str(OUT_DIR / 'r4c_phase2_ckpt.h5')
FINAL_W      = str(OUT_DIR / 'r4c_final_weights.h5')
FINAL_LOG    = str(OUT_DIR / 'r4c_training_log.csv')
FINAL_REPORT = str(OUT_DIR / 'r4c_final_eval_report.json')

print(f'Input weights : {IN_WEIGHTS}')
print(f'Output dir    : {OUT_DIR}')

Mounted at /content/drive
Input weights : /content/drive/MyDrive/ATS_R4c/r4b_final_weights.h5
Output dir    : /content/drive/MyDrive/ATS_R4c/output


In [4]:
# Cell 4: Verify required files
required = [
    str(DATA_DIR / 'ats_tokenized.npz'),
    str(DATA_DIR / 'rsg_tokenized.npz'),
    IN_WEIGHTS,
    SPLITS_JSON,
]
all_ok = True
for f in required:
    exists   = os.path.exists(f)
    status   = 'OK     ' if exists else 'MISSING'
    size_str = f'{os.path.getsize(f)/1e6:.1f} MB' if exists else 'MISSING'
    print(f'  [{status}]  {size_str:>10}   {f}')
    all_ok = all_ok and exists
exists_csv = os.path.exists(MERGED_CSV)
csv_size   = f'{os.path.getsize(MERGED_CSV)/1e6:.1f} MB' if exists_csv else 'not uploaded'
print(f'  [OPTIONAL]  {csv_size:>10}   {MERGED_CSV}')
if not all_ok:
    raise FileNotFoundError('Missing required files -- upload to Drive first.')
print()
print('All required files present.')

  [OK     ]      3.0 MB   /content/drive/MyDrive/ATS_R4c/ats_tokenized.npz
  [OK     ]      0.1 MB   /content/drive/MyDrive/ATS_R4c/rsg_tokenized.npz
  [OK     ]     93.9 MB   /content/drive/MyDrive/ATS_R4c/r4b_final_weights.h5
  [OK     ]      0.1 MB   /content/drive/MyDrive/ATS_R4c/data_splits.json
  [OPTIONAL]     68.9 MB   /content/drive/MyDrive/ATS_R4c/merged_final.csv

All required files present.


In [5]:
# Cell 5: Imports & environment
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import csv, json
import numpy as np
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import f1_score

gpu_devices = tf.config.list_physical_devices('GPU')
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {gpu_devices}')

TensorFlow : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# Cell 6: Hyperparameters -- R4c config
MINILM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEQ_LEN           = 128

# ── IT upsampling ─────────────────────────────────────────────────────────────
IT_UPSAMPLE = 3   # 3x data exposure for IT class

# ── Phase 1 ─────────────────────────────────────────────────────────────────
# Phase 1 (frozen-encoder domain-head training) is SKIPPED.
# Root cause: IT/Finance boundary confusion lives in encoder embedding space.
# Frozen-encoder training cannot fix it and consistently regresses both classes.
# Phase 2 joint training allows encoder adaptation -- the only effective path.
P1_LR         = 5e-5   # kept for reference, not used
P1_BATCH      = 32
P1_MAX_EPOCHS = 0       # 0 = skip Phase 1
P1_PATIENCE   = 8
P1_TARGET_F1  = 0.85
P1_FOCAL_GAMMA = 1.5

P1_DOM_WEIGHTS = {
    0: 2.0, 1: 0.8, 2: 0.3, 3: 0.7, 4: 2.5, 5: 0.25, 6: 0.5,
}

# ── Phase 2: Joint polish ───────────────────────────────────────────────────
P2_LR          = 8e-6
P2_BATCH       = 32
P2_MAX_EPOCHS  = 30
P2_PATIENCE    = 8
P2_FOCAL_GAMMA = 1.5   # focal loss applied to domain head in Phase 2

W_ATS = 0.35
W_DOM = 0.35
W_RSG = 0.30

# Phase 2 domain class weights
# 3x IT upsample does data-exposure work; weights fine-tune gradient balance.
# Effective IT boost = 3 (upsample) x 2.0 (weight) = 6x.
P2_DOM_WEIGHTS = {
    0: 2.0,   # IT         -- failing; 6x effective boost
    1: 0.8,   # Non-IT     -- passing
    2: 0.3,   # Design     -- passing well, free gradient
    3: 0.7,   # Healthcare -- passing
    4: 2.5,   # Finance    -- failing; elevated
    5: 0.25,  # Legal      -- passing well, free gradient
    6: 0.5,   # Edu        -- passing
}

GRAD_CLIP  = 1.0
HARD_STOP  = 8.5

GATE_ATS_VAL  = 6.5
GATE_ATS_TEST = 7.0
GATE_DOM_MAC  = 0.85
GATE_DOM_CLS  = 0.80
GATE_RSG      = 0.65
GATE_FRESH    = 20.0

DOMAIN_NAMES       = ['IT', 'Non-IT', 'Design', 'Healthcare', 'Finance', 'Legal', 'Edu']
DOMAIN_HEAD_LAYERS = ['dom_dense1', 'dom_drop1', 'dom_dense2', 'dom_drop2', 'domain_probs']

FRESHER_KEYWORDS = [
    'fresher', 'entry level', 'entry-level', '0 years experience',
    '0-1 years', 'recent graduate', 'final year', 'fresh graduate',
    'no prior experience', 'no experience',
]

print('Hyperparameters configured.')
print(f'  IT upsampling : {IT_UPSAMPLE}x')
print(f'  Phase 1       : SKIPPED (frozen-encoder training regressed IT/Finance)')
print(f'  Phase 2       : lr={P2_LR}  focal_gamma={P2_FOCAL_GAMMA}  max_epochs={P2_MAX_EPOCHS}  patience={P2_PATIENCE}')
print(f'  Grad clip     : {GRAD_CLIP}  |  ATS/DOM/RSG: {W_ATS}/{W_DOM}/{W_RSG}')


Hyperparameters configured.
  IT upsampling : 3x
  Phase 1       : SKIPPED (frozen-encoder training regressed IT/Finance)
  Phase 2       : lr=8e-06  focal_gamma=1.5  max_epochs=30  patience=8
  Grad clip     : 1.0  |  ATS/DOM/RSG: 0.35/0.35/0.3


In [7]:
# Cell 7: MiniLM model definition
import transformers as _tf_check
_major, _minor = (int(x) for x in _tf_check.__version__.split('.')[:2])
if (_major, _minor) >= (4, 47):
    raise RuntimeError(
        f'transformers {_tf_check.__version__} detected -- TFAutoModel removed in 4.47. '
        'Run Cell 2, then Runtime -> Restart session -> re-run from Cell 1.'
    )

from transformers import TFAutoModel


def mean_pool_l2(last_hidden, attention_mask):
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        super().__init__(**kwargs)
        self._bert = bert_model

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=training,
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    x1        = tf.keras.layers.Dense(256, activation='relu',   name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                    name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',   name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                    name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid',  name='ats_score')(x1)

    x2           = tf.keras.layers.Dense(256, activation='relu',  name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                   name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu',  name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                   name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax', name='domain_probs')(x2)

    x3           = tf.keras.layers.Dense(512, activation='relu',  name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                   name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu',  name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu',  name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(            name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                   name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax', name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )


print(f'transformers {_tf_check.__version__} OK.')
print('Model definition ready.')

transformers 4.44.2 OK.
Model definition ready.


In [8]:
# Cell 8: Load tokenized data & splits
print('Loading tokenized data...')
ats_data = np.load(str(DATA_DIR / 'ats_tokenized.npz'))
rsg_data = np.load(str(DATA_DIR / 'rsg_tokenized.npz'))

with open(SPLITS_JSON) as f:
    splits = json.load(f)

ats_tr_idx  = np.array(splits['ats_train'])
ats_vl_idx  = np.array(splits['ats_val'])
ats_tst_idx = np.array(splits['ats_test'])
rsg_tr_idx  = np.array(splits['rsg_train'])
rsg_vl_idx  = np.array(splits['rsg_val'])

ATS_KEYS = ('resume_input_ids', 'resume_attention_mask',
             'jd_input_ids', 'jd_attention_mask', 'ats_scores', 'domain_labels')
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_tr  = {k: ats_data[k][ats_tr_idx]  for k in ATS_KEYS}
ats_vl  = {k: ats_data[k][ats_vl_idx]  for k in ATS_KEYS}
ats_tst = {k: ats_data[k][ats_tst_idx] for k in ATS_KEYS}
rsg_tr  = {k: rsg_data[k][rsg_tr_idx]  for k in RSG_KEYS}
rsg_vl  = {k: rsg_data[k][rsg_vl_idx]  for k in RSG_KEYS}

n_ats_tr = len(ats_tr_idx)
n_rsg_tr = len(rsg_tr_idx)
print(f'  ATS  train={n_ats_tr:,}  val={len(ats_vl_idx):,}  test={len(ats_tst_idx):,}')
print(f'  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}')

Loading tokenized data...
  ATS  train=6,702  val=1,340  test=894
  RSG  train=1,560  val=390


In [9]:
# Cell 9: IT upsampling -- build augmented training set
# Duplicate IT (class 0) samples IT_UPSAMPLE times.
# Validation and test sets are NOT modified; only training set is augmented.

it_mask  = ats_tr['domain_labels'] == 0
n_it     = int(it_mask.sum())
n_non_it = n_ats_tr - n_it

print(f'Original training set : {n_ats_tr:,} samples')
print(f'  IT samples          : {n_it:,}  ({n_it/n_ats_tr:.1%})')
print(f'  Non-IT samples      : {n_non_it:,}  ({n_non_it/n_ats_tr:.1%})')
print(f'IT upsampling factor  : {IT_UPSAMPLE}x')

ats_tr_aug = {}
for k in ATS_KEYS:
    it_part = ats_tr[k][it_mask]
    ats_tr_aug[k] = np.concatenate(
        [ats_tr[k]] + [it_part] * (IT_UPSAMPLE - 1), axis=0
    )

n_ats_tr_aug = len(ats_tr_aug['ats_scores'])
n_it_aug     = int((ats_tr_aug['domain_labels'] == 0).sum())

print(f'Augmented training set: {n_ats_tr_aug:,} samples')
print(f'  IT samples (aug)    : {n_it_aug:,}  ({n_it_aug/n_ats_tr_aug:.1%})')

p1_dom_w_aug = np.array(
    [P1_DOM_WEIGHTS.get(int(d), 1.0) for d in ats_tr_aug['domain_labels']],
    dtype='float32'
)
p2_dom_w_aug = np.array(
    [P2_DOM_WEIGHTS.get(int(d), 1.0) for d in ats_tr_aug['domain_labels']],
    dtype='float32'
)
print('Phase 1 and Phase 2 sample weights computed on augmented set.')

Original training set : 6,702 samples
  IT samples          : 875  (13.1%)
  Non-IT samples      : 5,827  (86.9%)
IT upsampling factor  : 3x
Augmented training set: 8,452 samples
  IT samples (aug)    : 2,625  (31.1%)
Phase 1 and Phase 2 sample weights computed on augmented set.


In [10]:
# Cell 10: Build model & load input weights
print(f'Loading MiniLM ({MINILM_MODEL_NAME})...')
bert_model = TFAutoModel.from_pretrained(MINILM_MODEL_NAME, from_pt=True)
bert_model.trainable = True

model = build_unified_minilm_model(bert_model)
model.load_weights(IN_WEIGHTS)
print(f'Weights loaded: {IN_WEIGHTS}')
print(f'Total params  : {model.count_params():,}')

Loading MiniLM (sentence-transformers/all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Weights loaded: /content/drive/MyDrive/ATS_R4c/r4b_final_weights.h5
Total params  : 23,430,326


In [11]:
# Cell 11: Eval helpers (shared across both phases)
_sce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def eval_ats(data, batch=64):
    n = len(data['ats_scores'])
    ats_p, dom_p = [], []
    for s in range(0, n, batch):
        e = min(s + batch, n)
        ap, dp, _ = model(
            [data['resume_input_ids'][s:e], data['resume_attention_mask'][s:e],
             data['jd_input_ids'][s:e],     data['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_p.append(ap.numpy())
        dom_p.append(dp.numpy())
    ats_p = np.concatenate(ats_p).squeeze(-1)
    dom_p = np.concatenate(dom_p)

    mae_01       = float(np.mean(np.abs(ats_p - data['ats_scores'])))
    mae_100      = mae_01 * 100.0
    dom_true     = data['domain_labels']
    dom_pred_cls = np.argmax(dom_p, 1)
    dom_f1_macro = f1_score(dom_true, dom_pred_cls, average='macro',  zero_division=0)
    per_cls_f1   = f1_score(dom_true, dom_pred_cls, average=None,
                            labels=list(range(7)), zero_division=0)
    dom_ce       = float(_sce(dom_true.astype('int32'), dom_p).numpy().mean())
    return mae_100, dom_f1_macro, per_cls_f1, dom_ce, mae_01


def eval_rsg(data, batch=64):
    m = len(data['rsg_labels'])
    rsg_p = []
    for s in range(0, m, batch):
        e    = min(s + batch, m)
        pids = data['profile_input_ids'][s:e]
        pmsk = data['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_p.append(rp.numpy())
    rsg_p      = np.concatenate(rsg_p)
    labels     = data['rsg_labels']
    top1_acc   = float(np.mean(np.argmax(rsg_p, 1) == labels))
    top3_preds = np.argsort(rsg_p, axis=1)[:, -3:]
    top3_acc   = float(np.mean([labels[i] in top3_preds[i] for i in range(m)]))
    rsg_ce     = float(_sce(labels.astype('int32'), rsg_p).numpy().mean())
    return top1_acc, top3_acc, rsg_ce


print('Eval helpers ready.')

Eval helpers ready.


In [12]:
# Cell 12: Baseline -- R4b output metrics (starting point for R4c)
print('Measuring input-weights baseline (val set)...')
b_mae, b_dom_f1, b_per_cls, _, _ = eval_ats(ats_vl)
b_rsg_acc, b_rsg_top3, _         = eval_rsg(rsg_vl)

print(f'  val ATS MAE  : {b_mae:.2f}   (gate < {GATE_ATS_VAL})')
print(f'  val Dom F1   : {b_dom_f1:.4f}  (gate > {GATE_DOM_MAC})')
print(f'  val RSG Top-1: {b_rsg_acc:.4f}  (gate > {GATE_RSG})')
print(f'  val RSG Top-3: {b_rsg_top3:.4f}')
print()
print('  Per-domain F1 (baseline from R4b weights):')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, b_per_cls)):
    flag = 'PASS' if f1v > GATE_DOM_CLS else 'FAIL <-- must fix'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}')

Measuring input-weights baseline (val set)...
  val ATS MAE  : 6.35   (gate < 6.5)
  val Dom F1   : 0.8369  (gate > 0.85)
  val RSG Top-1: 0.7846  (gate > 0.65)
  val RSG Top-3: 0.9821

  Per-domain F1 (baseline from R4b weights):
    [0] IT          : 0.7968  FAIL <-- must fix
    [1] Non-IT      : 0.8059  PASS
    [2] Design      : 0.9206  PASS
    [3] Healthcare  : 0.8506  PASS
    [4] Finance     : 0.7862  FAIL <-- must fix
    [5] Legal       : 0.8824  PASS
    [6] Edu         : 0.8154  PASS


In [13]:
# Cell 13: Phase 1 -- SKIPPED
# Frozen-encoder domain-head training regressed IT and Finance in all three
# R4c attempts. The IT/Finance boundary confusion is in the encoder embedding
# space; the domain head alone cannot fix it. Phase 2 joint training (all
# layers unfrozen) is the correct path.

print('Phase 1: SKIPPED -- proceeding directly to Phase 2.')
print()

# Initialise variables expected by Cell 14 and Cell 15
best_p1_f1    = b_dom_f1
best_p1_worst = min(b_per_cls)
p1_log        = []

# Save r4b baseline weights as the Phase 1 handoff so Phase 2 starts clean
model.save_weights(P1_BEST)
print(f'  r4b baseline weights saved -> P1_BEST (clean handoff to Phase 2)')
print(f'  Baseline DomF1: {best_p1_f1:.4f}  worst_cls: {best_p1_worst:.4f}')
print(f'  IT={b_per_cls[0]:.4f}  Fin={b_per_cls[4]:.4f}')


Phase 1: SKIPPED -- proceeding directly to Phase 2.

  r4b baseline weights saved -> P1_BEST (clean handoff to Phase 2)
  Baseline DomF1: 0.8369  worst_cls: 0.7862
  IT=0.7968  Fin=0.7862


In [14]:
# Cell 14: Phase 1 results summary
print('Phase 1 end-state (val):')
p1_mae, p1_dom_f1, p1_per_cls, _, _ = eval_ats(ats_vl)
p1_rsg_acc, p1_rsg_top3, _          = eval_rsg(rsg_vl)

print(f'  val ATS MAE  : {p1_mae:.2f}   baseline was {b_mae:.2f}')
print(f'  val Dom F1   : {p1_dom_f1:.4f}  baseline was {b_dom_f1:.4f}  delta={p1_dom_f1-b_dom_f1:+.4f}')
print(f'  val RSG Top-1: {p1_rsg_acc:.4f}  baseline was {b_rsg_acc:.4f}')
print()
print('  Per-domain F1 after Phase 1:')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, p1_per_cls)):
    delta = f1v - b_per_cls[i]
    flag  = 'PASS' if f1v > GATE_DOM_CLS else 'FAIL'
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}  (delta {delta:+.4f})')

Phase 1 end-state (val):
  val ATS MAE  : 6.35   baseline was 6.35
  val Dom F1   : 0.8369  baseline was 0.8369  delta=+0.0000
  val RSG Top-1: 0.7846  baseline was 0.7846

  Per-domain F1 after Phase 1:
    [0] IT          : 0.7968  FAIL  (delta +0.0000)
    [1] Non-IT      : 0.8059  PASS  (delta +0.0000)
    [2] Design      : 0.9206  PASS  (delta +0.0000)
    [3] Healthcare  : 0.8506  PASS  (delta +0.0000)
    [4] Finance     : 0.7862  FAIL  (delta +0.0000)
    [5] Legal       : 0.8824  PASS  (delta +0.0000)
    [6] Edu         : 0.8154  PASS  (delta +0.0000)


In [15]:
# Cell 15: -- PHASE 2: Joint polish (focal loss + worst-class checkpoint)
print('=' * 65)
print('  PHASE 2: Joint polish -- encoder + all heads unfrozen')
print(f'  LR={P2_LR}  clipnorm={GRAD_CLIP}  focal_gamma={P2_FOCAL_GAMMA}')
print(f'  Checkpoint metric: worst-class F1 (directly targets IT & Finance)')
print(f'  Training on {n_ats_tr_aug:,} samples ({IT_UPSAMPLE}x IT upsampled)')
print('=' * 65)

for layer in model.layers:
    layer.trainable = True
bert_model.trainable = True

trainable_p2 = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
print(f'  Trainable params: {trainable_p2:,} (all layers)')
print()

opt_p2   = tf.keras.optimizers.Adam(learning_rate=P2_LR, clipnorm=GRAD_CLIP)
_mae_fn  = tf.keras.losses.MeanAbsoluteError()
_sce_rsg = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def focal_dom_loss_p2(y_true, y_pred, gamma=1.5, sample_weight=None):
    """Focal loss: concentrates domain gradient on IT/Finance where
    the model is uncertain; reduces gradient from easy classes (Design, Legal)."""
    y_true_int = tf.cast(y_true, tf.int32)
    ce    = tf.keras.losses.sparse_categorical_crossentropy(y_true_int, y_pred)
    p_t   = tf.reduce_sum(y_pred * tf.one_hot(y_true_int, 7), axis=-1)
    fl    = tf.pow(1.0 - p_t, gamma) * ce
    if sample_weight is not None:
        fl = fl * tf.cast(sample_weight, tf.float32)
    return tf.reduce_mean(fl)


def train_step_ats_dom(r_ids, r_mask, jd_ids, jd_mask, ats_y, dom_y, dom_w):
    with tf.GradientTape() as tape:
        ats_p, dom_p, _ = model([r_ids, r_mask, jd_ids, jd_mask], training=True)
        ats_l = _mae_fn(tf.expand_dims(ats_y, 1), ats_p)
        dom_l = focal_dom_loss_p2(dom_y, dom_p, gamma=P2_FOCAL_GAMMA, sample_weight=dom_w)
        total = W_ATS * ats_l + W_DOM * dom_l
    opt_p2.apply_gradients(
        zip(tape.gradient(total, model.trainable_variables), model.trainable_variables)
    )
    return float(ats_l), float(dom_l)


def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        loss = W_RSG * _sce_rsg(rsg_y, rsg_p)
    opt_p2.apply_gradients(
        zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables)
    )
    return float(loss)


best_p2_dom_f1 = best_p1_f1     # macro F1 (for reporting)
best_p2_worst  = best_p1_worst  # worst-class F1 (checkpoint metric)
best_p2_epoch  = 0
p2_patience    = 0
p2_log         = []
hard_stop      = False
n_ats_batches  = -(-n_ats_tr_aug // P2_BATCH)

model.load_weights(P1_BEST)   # ensure clean r4b baseline
model.save_weights(P2_BEST)

print(f'  ATS batches/epoch : {n_ats_batches}  |  RSG train: {n_rsg_tr:,}')
print(f'  Checkpoint threshold: worst_cls > {best_p2_worst:.4f}')
print(f'  Baseline: IT={b_per_cls[0]:.4f}  Fin={b_per_cls[4]:.4f}  worst={best_p2_worst:.4f}')
print()

rng = np.random.default_rng(42)  # <-- Add this line right here
for epoch in range(1, P2_MAX_EPOCHS + 1):

    ats_perm = rng.permutation(n_ats_tr_aug)
    rsg_perm = rng.permutation(n_rsg_tr)
    rsg_pool = np.tile(rsg_perm, (n_ats_batches // max(1, n_rsg_tr // P2_BATCH) + 2))
    rsg_cur  = 0

    ats_ls, dom_ls, rsg_ls = [], [], []

    for bi in range(n_ats_batches):
        idx_a = ats_perm[bi * P2_BATCH : (bi + 1) * P2_BATCH]
        al, dl = train_step_ats_dom(
            ats_tr_aug['resume_input_ids'][idx_a],
            ats_tr_aug['resume_attention_mask'][idx_a],
            ats_tr_aug['jd_input_ids'][idx_a],
            ats_tr_aug['jd_attention_mask'][idx_a],
            ats_tr_aug['ats_scores'][idx_a].astype('float32'),
            ats_tr_aug['domain_labels'][idx_a].astype('int32'),
            p2_dom_w_aug[idx_a],
        )
        ats_ls.append(al)
        dom_ls.append(dl)

        idx_r   = rsg_pool[rsg_cur * P2_BATCH : (rsg_cur + 1) * P2_BATCH]
        idx_r   = idx_r[:P2_BATCH]
        rsg_cur = (rsg_cur + 1) % max(1, n_rsg_tr // P2_BATCH)
        rl = train_step_rsg(
            rsg_tr['profile_input_ids'][idx_r],
            rsg_tr['profile_attention_mask'][idx_r],
            rsg_tr['rsg_labels'][idx_r].astype('int32'),
        )
        rsg_ls.append(rl)

    if np.isnan(float(np.mean(ats_ls))):
        print()
        print(f'HARD STOP -- NaN loss at epoch {epoch}')
        model.load_weights(P2_BEST)
        hard_stop = True
        break

    val_mae, val_dom_f1, val_per_cls, _, _ = eval_ats(ats_vl)
    val_rsg_acc, val_rsg_top3, _           = eval_rsg(rsg_vl)
    val_worst_cls = float(min(val_per_cls))
    it_f1         = float(val_per_cls[0])
    fin_f1        = float(val_per_cls[4])

    print(
        f'  P2 Ep {epoch:02d}/{P2_MAX_EPOCHS} | '
        f'tr_ATS:{np.mean(ats_ls):.4f}  '
        f'val_MAE:{val_mae:.2f}  '
        f'DomF1:{val_dom_f1:.4f}  '
        f'IT:{it_f1:.4f}  Fin:{fin_f1:.4f}  '
        f'worst:{val_worst_cls:.4f}  RSG:{val_rsg_acc:.3f}'
    )

    p2_log.append({
        'phase': 2, 'epoch': epoch,
        'train_ats_mae':  round(float(np.mean(ats_ls)), 6),
        'train_dom_loss': round(float(np.mean(dom_ls)), 6),
        'train_rsg_loss': round(float(np.mean(rsg_ls)), 6),
        'val_mae_100':    round(val_mae, 4),
        'val_dom_f1':     round(val_dom_f1, 4),
        'val_rsg_acc':    round(val_rsg_acc, 4),
        'it_f1':          round(it_f1, 4),
        'finance_f1':     round(fin_f1, 4),
        'worst_cls_f1':   round(val_worst_cls, 4),
    })

    if val_mae > HARD_STOP:
        print()
        print(f'  *** HARD STOP epoch {epoch}: val MAE {val_mae:.2f} > {HARD_STOP}. Restoring best. ***')
        model.load_weights(P2_BEST)
        hard_stop = True
        break

    # Checkpoint on worst-class F1 so IT and Finance gates drive saving
    if val_worst_cls > best_p2_worst:
        best_p2_worst  = val_worst_cls
        best_p2_dom_f1 = val_dom_f1
        best_p2_epoch  = epoch
        p2_patience    = 0
        model.save_weights(P2_BEST)
        it_s  = 'PASS' if it_f1  > GATE_DOM_CLS else 'fail'
        fin_s = 'PASS' if fin_f1 > GATE_DOM_CLS else 'fail'
        print(f'    -> New best worst_cls={val_worst_cls:.4f}  DomF1={val_dom_f1:.4f}  IT:{it_s}  Fin:{fin_s} -- saved.')
    else:
        p2_patience += 1
        print(f'    -> No improvement ({p2_patience}/{P2_PATIENCE})')
        if p2_patience >= P2_PATIENCE:
            print()
            print(f'  Phase 2 early stop at epoch {epoch}.')
            model.load_weights(P2_BEST)
            break

print()
print(f'Phase 2 complete. Best worst_cls: {best_p2_worst:.4f}  DomF1: {best_p2_dom_f1:.4f} at epoch {best_p2_epoch}')


  PHASE 2: Joint polish -- encoder + all heads unfrozen
  LR=8e-06  clipnorm=1.0  focal_gamma=1.5
  Checkpoint metric: worst-class F1 (directly targets IT & Finance)
  Training on 8,452 samples (3x IT upsampled)
  Trainable params: 23,428,534 (all layers)

  ATS batches/epoch : 265  |  RSG train: 1,560
  Checkpoint threshold: worst_cls > 0.7862
  Baseline: IT=0.7968  Fin=0.7862  worst=0.7862

  P2 Ep 01/30 | tr_ATS:0.0472  val_MAE:6.31  DomF1:0.8197  IT:0.7629  Fin:0.7413  worst:0.7413  RSG:0.792
    -> No improvement (1/8)
  P2 Ep 02/30 | tr_ATS:0.0467  val_MAE:6.38  DomF1:0.8285  IT:0.7772  Fin:0.7751  worst:0.7751  RSG:0.795
    -> No improvement (2/8)
  P2 Ep 03/30 | tr_ATS:0.0461  val_MAE:6.31  DomF1:0.8339  IT:0.7855  Fin:0.7682  worst:0.7682  RSG:0.787
    -> No improvement (3/8)
  P2 Ep 04/30 | tr_ATS:0.0454  val_MAE:6.36  DomF1:0.8313  IT:0.7835  Fin:0.7685  worst:0.7685  RSG:0.785
    -> No improvement (4/8)
  P2 Ep 05/30 | tr_ATS:0.0448  val_MAE:6.30  DomF1:0.8330  IT:0.7874

In [ ]:
# Cell 16: Save training log (both phases)
all_log = p1_log + p2_log
if all_log:
    with open(FINAL_LOG, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=all_log[0].keys())
        writer.writeheader()
        writer.writerows(all_log)
    print(f'Training log saved -> {FINAL_LOG}')

In [ ]:
# Cell 17: Save final weights
model.load_weights(P2_BEST)
model.save_weights(FINAL_W)
print(f'Final weights saved -> {FINAL_W}')

In [ ]:
# Cell 18: Full test + val evaluation
print('Running full evaluation...')

test_mae, test_dom_f1, test_per_cls, _, _ = eval_ats(ats_tst)
val_mae_f, val_dom_f1_f, val_per_cls_f, _, _ = eval_ats(ats_vl)
rsg_top1, rsg_top3, _ = eval_rsg(rsg_vl)

print(f'  ATS val  MAE : {val_mae_f:.2f}')
print(f'  ATS test MAE : {test_mae:.2f}')
print(f'  Dom test F1  : {test_dom_f1:.4f}')
print(f'  RSG val Top-1: {rsg_top1:.4f}')
print(f'  RSG val Top-3: {rsg_top3:.4f}')

In [ ]:
# Cell 19: Fresher fairness check
fresher_gap    = None
n_fresher      = 0
n_experienced  = 0
fresher_status = 'SKIPPED'

if os.path.exists(MERGED_CSV):
    print('Loading merged_final.csv...')
    import pandas as pd
    df = pd.read_csv(MERGED_CSV)
    if len(df) > max(ats_tst_idx):
        test_resumes = df.iloc[ats_tst_idx]['resume_text'].fillna('').values
        is_fresher   = np.array(
            [any(kw in r.lower() for kw in FRESHER_KEYWORDS) for r in test_resumes]
        )
        n_fresher     = int(is_fresher.sum())
        n_experienced = len(is_fresher) - n_fresher
        if n_fresher >= 10:
            all_scores = []
            n_tst = len(ats_tst_idx)
            for s in range(0, n_tst, 64):
                e = min(s + 64, n_tst)
                ap, _, _ = model(
                    [ats_tst['resume_input_ids'][s:e],
                     ats_tst['resume_attention_mask'][s:e],
                     ats_tst['jd_input_ids'][s:e],
                     ats_tst['jd_attention_mask'][s:e]], training=False)
                all_scores.append(ap.numpy().flatten())
            all_scores  = np.concatenate(all_scores) * 100.0
            fresher_gap = float(all_scores[~is_fresher].mean()) - float(all_scores[is_fresher].mean())
            fresher_status = 'PASS' if fresher_gap <= GATE_FRESH else 'FAIL'
            print(f'  Fresher: {n_fresher}  Experienced: {n_experienced}')
            print(f'  Gap    : {fresher_gap:.1f} pts  {fresher_status}')
        else:
            fresher_status = f'N/A (only {n_fresher} fresher samples)'
            print(f'  {fresher_status}')
    else:
        fresher_status = 'SKIPPED (index mismatch)'
        print(f'  {fresher_status}')
else:
    print('  merged_final.csv not found -- fresher gate skipped.')

In [ ]:
# Cell 20: Gate report
g_ats_val  = val_mae_f   < GATE_ATS_VAL
g_ats_test = test_mae    < GATE_ATS_TEST
g_dom_mac  = test_dom_f1 > GATE_DOM_MAC
g_dom_cls  = all(f > GATE_DOM_CLS for f in test_per_cls)
g_rsg      = rsg_top1    > GATE_RSG
g_fresh    = (fresher_gap is None) or (fresher_gap <= GATE_FRESH)
all_pass   = g_ats_val and g_ats_test and g_dom_mac and g_dom_cls and g_rsg and g_fresh


def pf(gate):
    return 'PASS' if gate else 'FAIL'


print()
print('=' * 65)
print('  INJECTION-R4c -- FINAL GATE RESULTS')
print('=' * 65)
print(f'  ATS val  MAE (0-100): {val_mae_f:.2f}   {pf(g_ats_val)}  (gate < {GATE_ATS_VAL})')
print(f'  ATS test MAE (0-100): {test_mae:.2f}   {pf(g_ats_test)}  (gate < {GATE_ATS_TEST})')
print(f'  Domain F1 macro     : {test_dom_f1:.4f}  {pf(g_dom_mac)}  (gate > {GATE_DOM_MAC})')
print(f'  RSG val Top-1 Acc   : {rsg_top1:.4f}  {pf(g_rsg)}  (gate > {GATE_RSG})')
print(f'  RSG val Top-3 Acc   : {rsg_top3:.4f}')
print()
print(f'  Per-domain F1 (gate > {GATE_DOM_CLS} each):')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, test_per_cls)):
    delta = f1v - b_per_cls[i]
    flag  = pf(f1v > GATE_DOM_CLS)
    print(f'    [{i}] {name:12s}: {f1v:.4f}  {flag}  (delta from R4b: {delta:+.4f})')
print()
if fresher_gap is not None:
    print(f'  Fresher gap : {fresher_gap:.1f} pts  {pf(g_fresh)}  (gate <= {GATE_FRESH})')
    print(f'    Fresher: {n_fresher}  |  Experienced: {n_experienced}')
else:
    print(f'  Fresher gap : {fresher_status}')
print()
print(f'  Phase 1 best DomF1  : {best_p1_f1:.4f}')
print(f'  Phase 2 best DomF1  : {best_p2_dom_f1:.4f} (epoch {best_p2_epoch})')
hs_str = 'YES' if hard_stop else 'NO'
print(f'  Hard stop triggered : {hs_str}')
print(f'  Output              : {FINAL_W}')
print('=' * 65)

if all_pass:
    print()
    print('R4c PASSED ALL GATES -- proceed to T1')
else:
    print()
    print('*** R4c FAILED one or more gates -- DO NOT proceed to T1. ***')

In [ ]:
# Cell 21: Save r4c_final_eval_report.json
report = {
    'stage': 'R4c',
    'all_gates_pass': bool(all_pass),
    'hard_stop': hard_stop,
    'phase1': {'best_dom_f1': round(best_p1_f1, 4)},
    'phase2': {'best_dom_f1': round(best_p2_dom_f1, 4), 'best_epoch': best_p2_epoch},
    'gates': {
        'ats_val_mae_100':     {'value': round(val_mae_f, 4),   'threshold': f'< {GATE_ATS_VAL}',  'pass': bool(g_ats_val)},
        'ats_test_mae_100':    {'value': round(test_mae, 4),    'threshold': f'< {GATE_ATS_TEST}', 'pass': bool(g_ats_test)},
        'domain_f1_macro':     {'value': round(test_dom_f1, 4), 'threshold': f'> {GATE_DOM_MAC}',  'pass': bool(g_dom_mac)},
        'domain_all_cls_pass': bool(g_dom_cls),
        'rsg_val_top1':        {'value': round(rsg_top1, 4),    'threshold': f'> {GATE_RSG}',      'pass': bool(g_rsg)},
        'rsg_val_top3':        round(rsg_top3, 4),
        'fresher_gap':         fresher_gap if fresher_gap is not None else fresher_status,
        'fresher_pass':        bool(g_fresh),
    },
    'per_domain_f1': {
        DOMAIN_NAMES[i]: {
            'f1':             round(float(test_per_cls[i]), 4),
            'pass':           bool(test_per_cls[i] > GATE_DOM_CLS),
            'delta_from_r4b': round(float(test_per_cls[i] - b_per_cls[i]), 4),
        }
        for i in range(7)
    },
    'baseline_r4b': {
        'val_mae': round(b_mae, 4),
        'dom_f1':  round(b_dom_f1, 4),
        'rsg_acc': round(b_rsg_acc, 4),
    },
    'training_config': {
        'it_upsample': IT_UPSAMPLE,
        'phase1': {'lr': P1_LR, 'max_epochs': P1_MAX_EPOCHS, 'patience': P1_PATIENCE,
                   'dom_weights': P1_DOM_WEIGHTS},
        'phase2': {'lr': P2_LR, 'max_epochs': P2_MAX_EPOCHS, 'patience': P2_PATIENCE,
                   'loss_weights': {'ats': W_ATS, 'dom': W_DOM, 'rsg': W_RSG},
                   'dom_weights': P2_DOM_WEIGHTS, 'grad_clip': GRAD_CLIP},
    },
}

with open(FINAL_REPORT, 'w') as fh:
    json.dump(report, fh, indent=2)
print(f'Report saved -> {FINAL_REPORT}')

In [ ]:
# Cell 22: Download results
from google.colab import files

print('Downloading r4c_final_weights.h5 ...')
files.download(FINAL_W)

print('Downloading r4c_final_eval_report.json ...')
files.download(FINAL_REPORT)

print('Downloading r4c_training_log.csv ...')
files.download(FINAL_LOG)

print()
print('Save to local project:')
print('  r4c_final_weights.h5       -> model/unified_model/r4c_final_weights.h5')
print('  r4c_final_eval_report.json -> evaluation/r4c_final_eval_report.json')